# Fitbit activity (steps, floors, activityCalories) * microbe presence / absence interaction effect models

In [1]:
import warnings
from arivale_data_interface import get_snapshot
import pandas as pd
import numpy as np
import scipy
import json

warnings.simplefilter("ignore")

#import the final master sleep_merged df and the features_dict

sleep_merged = pd.read_csv('../working_df/sleep_merged_sleep_meds_excluded_10_14_2025.csv', dtype={'public_client_id': object})

with open('../working_df/features_dict.json', 'r') as f:
    features_dict = json.load(f)

In [2]:
sleep_merged['fitbit_dip_minus_reference_microbe_dip'] = sleep_merged.fitbit_dip - sleep_merged.reference_microbe_dip

In [3]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
from sklearn.metrics import r2_score

def interaction_regression_results(
    dependent_features,
    interaction_features,  # list of tuples, e.g., [("activityA", "microbeA")]
    drop_subset_list,
    df,
    test_df=None,
    train_test=False,
    additional_covariates=False
):
    all_results = []
    error_log = []

    for dependent_feature in dependent_features:
        for interaction_pair in interaction_features:
            # main1 will be activity, main2 will be binarized microbe
            main1, main2 = interaction_pair
            interaction_term = f"{main1}:C({main2})"
            formula = f"{dependent_feature} ~ {main1} + C({main2}) + {interaction_term} + C(sex) + age + BMI_CALC + PC1 + PC2 + PC3 + microbiome_vendor + fitbit_dip_minus_reference_microbe_dip + avg_stress + any_mental_health_self_anytime_before + any_sleep_disorder_self_anytime_before + mental_health_gut_sleep_q_vendor"

            # Drop rows with missing data
            drop_cols = [dependent_feature, main1, main2] + drop_subset_list
            df_dropped_nan = df.dropna(subset=drop_cols)
            if test_df is not None:
                test_df_dropped_nan = test_df.dropna(subset=drop_cols)

            # Check to make sure binary microbe is between 10-90% of total N, otherwise log error and skip
            if (df_dropped_nan[main2].mean() < 0.1) | (df_dropped_nan[main2].mean() > 0.9):
                error_log.append({
                    "dependent_feature": dependent_feature,
                    "independent_feature": interaction_term,
                    "formula": formula,
                    "error": "Too few samples",
                    "model_type": "linear"
                })
                continue

            try:
                model = ols(formula, data=df_dropped_nan)
                fitted = model.fit()

                # Compute test R2 on test df if requested
                if train_test:
                    y_test = test_df_clean[dependent_feature]
                    predictions = fitted.predict(test_df_clean)
                    r2_test = r2_score(y_test, predictions)
                else:
                    r2_test = np.nan

                result = {
                    "dependent_feature": dependent_feature,
                    "main1": main1,
                    "main2": main2,
                    "interaction": interaction_term,
                    "beta_main1": fitted.params.get(main1, np.nan),
                    "p_main1": fitted.pvalues.get(main1, np.nan),
                    # Use different syntax bc main2 has C() around it
                    "beta_main2": fitted.params[[i for i in fitted.params.index if main2 in i and main1 not in i]][0],
                    "p_main2": fitted.pvalues[[i for i in fitted.pvalues.index if main2 in i and main1 not in i]][0],
                    "beta_interaction": fitted.params[[i for i in fitted.params.index if main1 in i and main2 in i]][0],
                    "p_interaction": fitted.pvalues[[i for i in fitted.pvalues.index if main1 in i and main2 in i]][0],
                    "r2_train": fitted.rsquared,
                    "r2_test": r2_test,
                    "n_train": fitted.nobs,
                    "n_test": len(test_df_dropped_nan) if train_test else np.nan,
                    "formula": formula
                }

                all_results.append(pd.Series(result))

            except Exception as e:
                error_log.append({
                    "dependent_feature": dependent_feature,
                    "independent_feature": interaction_term,
                    "formula": formula,
                    "error": str(e),
                    "model_type": "linear"
                })

    return pd.DataFrame(all_results), pd.DataFrame(error_log)

## Define all the interaction terms

In [3]:
activity_features = ["activities_activityCalories_resid", "activities_steps_resid", "activities_floors_resid"]

In [5]:
from itertools import product

# Make tuples for every interaction we're interested in
# So this would be activity_features * microbe_features_binary and heartrate_features * microbe_features_binary
interaction_features_activity = (
    list(product(activity_features, features_dict['new_microbe_binarized_features']))
)

In [6]:
interaction_features_activity

[('activities_activityCalories_resid',
  'Archaea_Euryarchaeota_Methanobacteria_Methanobacteriales_Methanobacteriaceae_Methanobrevibacter_binarized'),
 ('activities_activityCalories_resid',
  'Archaea_Euryarchaeota_Methanobacteria_Methanobacteriales_Methanobacteriaceae_Methanosphaera_binarized'),
 ('activities_activityCalories_resid',
  'Archaea_Euryarchaeota_Methanobacteria_Methanobacteriales_Methanobacteriaceae_nan_binarized'),
 ('activities_activityCalories_resid',
  'Archaea_Euryarchaeota_Thermoplasmata_Methanomassiliicoccales_Methanomassiliicoccaceae_Methanomassiliicoccus_binarized'),
 ('activities_activityCalories_resid',
  'Archaea_Euryarchaeota_Thermoplasmata_Methanomassiliicoccales_Methanomethylophilaceae_Candidatus_Methanogranum_binarized'),
 ('activities_activityCalories_resid',
  'Archaea_Euryarchaeota_Thermoplasmata_Methanomassiliicoccales_Methanomethylophilaceae_Candidatus_Methanomethylophilus_binarized'),
 ('activities_activityCalories_resid',
  'Archaea_Euryarchaeota_Th

In [7]:
len(interaction_features_activity)

1800

In [12]:
1800*18 # This is the total regressions to fit, bc there are 18 different sleep features

32400

## Fit adjusted interaction models

In [20]:
columns_to_drop = ['sex', 'age', 'BMI_CALC', 'PC1', 'PC2', 'PC3', 'microbiome_vendor', 'fitbit_dip_minus_reference_microbe_dip',
                   'avg_stress', 'any_mental_health_self_anytime_before',
                   'any_sleep_disorder_self_anytime_before', 'mental_health_gut_sleep_q_vendor']

In [21]:
sleep_merged.dropna(subset=columns_to_drop).shape

(1469, 5897)

In [22]:
results_activity, error_log_activity = interaction_regression_results(
    features_dict['sleep_features_log1p_resids'],
    interaction_features_activity,  # list of tuples, e.g., [("activityA", "microbeA")]
    columns_to_drop,
    sleep_merged,
    test_df=None,
    train_test=False,
    additional_covariates=True
)

from statsmodels.stats.multitest import multipletests
# Apply FDR correction
results_activity = results_activity.groupby(['dependent_feature', 'main1']).apply(
    lambda group: group.assign(q=multipletests(group['p_interaction'], method='fdr_bh')[1])
).reset_index(drop=True)

# Make directories to save the analyses into
base_dir = "activity_x_microbe_interaction_sleep_adjusted_01_30_2026"
error_dir = os.path.join(base_dir, "error_outputs")

os.makedirs(error_dir, exist_ok=True)  # makes both folders if needed

results_activity.to_csv(
    os.path.join(base_dir, "activity_new_microbe_binary_interaction_results_adjusted_01-30-2026.csv"),
    index=False
)

error_log_activity.to_csv(
    os.path.join(error_dir, "activity_new_microbe_binary_interaction_adjusted_error_log_01-30-2026.csv"),
    index=False
)

# Visualize results of adjusted interaction models

In [12]:
df1 = pd.read_csv('activity_x_microbe_interaction_sleep_adjusted_01_30_2026/activity_new_microbe_binary_interaction_results_adjusted_01-30-2026.csv')
df1.rename(columns={'q': 'q_interaction'}, inplace=True)
df1.columns

Index(['dependent_feature', 'main1', 'main2', 'interaction', 'beta_main1',
       'p_main1', 'beta_main2', 'p_main2', 'beta_interaction', 'p_interaction',
       'r2_train', 'r2_test', 'n_train', 'n_test', 'formula', 'q_interaction'],
      dtype='object')

In [13]:
df1.shape

(8142, 16)

In [14]:
df1.n_train.describe()

count    8142.000000
mean     1063.500368
std       391.533349
min       497.000000
25%       898.000000
50%       898.000000
75%      1469.000000
max      1469.000000
Name: n_train, dtype: float64

In [15]:
df1[df1.q_interaction < 0.1].n_train.describe()

count      24.000000
mean     1040.750000
std       252.568071
min       898.000000
25%       898.000000
50%       898.000000
75%      1040.750000
max      1469.000000
Name: n_train, dtype: float64

In [16]:
df1.main2.nunique()

154

In [17]:
df1[df1.q_interaction < 0.1].main2.unique()

array(['Bacteria_Bacteroidetes_Bacteroidia_Bacteroidales_Barnesiellaceae_Coprobacter_binarized',
       'Bacteria_Firmicutes_Clostridia_Clostridiales_Family_XIII_Family_XIII_AD3011_group_binarized',
       'Bacteria_Firmicutes_Clostridia_Clostridiales_Lachnospiraceae_Shuttleworthia_binarized',
       'Bacteria_Firmicutes_Clostridia_Clostridiales_Ruminococcaceae_Oscillospira_binarized',
       'Bacteria_Firmicutes_Clostridia_Clostridiales_Ruminococcaceae_Pseudoflavonifractor_binarized',
       'Bacteria_Firmicutes_Clostridia_Clostridiales_Ruminococcaceae_Ruminococcaceae_UCG_005_binarized',
       'Bacteria_Firmicutes_Clostridia_Clostridiales_Ruminococcaceae_Ruminococcus_1_binarized',
       'Bacteria_Actinobacteria_Coriobacteriia_Coriobacteriales_Eggerthellaceae_Eggerthella_binarized',
       'Bacteria_Bacteroidetes_Bacteroidia_Bacteroidales_Muribaculaceae_nan_binarized',
       'Bacteria_Firmicutes_Clostridia_Clostridiales_Clostridiaceae_1_Clostridium_sensu_stricto_1_binarized',
      

In [18]:
# ------------------------------------------------------------------------------
# Make class-level taxonomic assignment column
# ------------------------------------------------------------------------------
to_plot_activity = df1[df1.q_interaction < 0.1]
to_plot_activity['class_tax'] = to_plot_activity.apply(
    lambda row: row['main2'].split('_')[2],
    axis=1
)

In [19]:
# ------------------------------------------------------------------------------
# 1. CLEAN MICROBE TAXONOMIC STRING (Family + Genus or next level)
# ------------------------------------------------------------------------------

# Make another cleaning function similar to the one we used to make the PyVis graph
def extract_family_and_clean2(feature: str) -> str:
    # Remove '_binarized'
    feature = feature.replace('_binarized', '').replace('_nan', '_NA')

    # Split into parts
    parts = feature.split('_')

    if 'Family_XIII' in feature:
        order = parts[3]
        return 'Clostridiales_Family_XIII_AD3011_group'
    elif 'Clostridiales_NA_NA' in feature:
        return 'Clostridiales_NA_NA'
    elif 'Clostridiales_vadinBB60_group' in feature:
        return 'Clostridiales_vadinBB60_group'
    elif parts[4] == parts[5]:
        return '_'.join(parts[5:])
    else:
        return '_'.join(parts[4:])

# ------------------------------------------------------------------------------
# 2. COMPUTE SLOPE ABSENT / PRESENT + SIGN LABELS
# ------------------------------------------------------------------------------

def slope_sign(x):
    """Return sign symbol for slope."""
    if x > 0:
        return "+"
    elif x < 0:
        return "−"
    else:
        return "0"


def annotate_interaction_slopes(df):
    """
    Adds slope_absent, slope_present, sign_absent, sign_present,
    effect_label, clean_name, annotated_name to df.
    """
    
    # Compute slopes
    df["slope_absent"]  = df["beta_main1_adjusted"]
    df["slope_present"] = df["beta_main1_adjusted"] + df["beta_interaction_adjusted"]
    
    # Convert slopes into sign symbols
    df["sign_absent"]  = df["slope_absent"].apply(slope_sign)
    df["sign_present"] = df["slope_present"].apply(slope_sign)
    
    # Create the effect label string
    df["effect_label"] = (
        "(absent: " + df["sign_absent"] + 
        ", present: " + df["sign_present"] + ")"
    )
    
    # Clean microbe name
    df["clean_name"] = df["main2"].apply(extract_family_and_clean2)
    
    # Annotated microbe name ready for Venn diagram
    df["annotated_name"] = df["clean_name"] + " " + df["effect_label"]
    
    return df

In [20]:
#Apply to independent_feature in final_df
to_plot_activity['main2'] = to_plot_activity.apply(
    lambda row: extract_family_and_clean2(row['main2']),
    axis=1
)

In [21]:
to_plot_activity.main2.unique()

array(['Barnesiellaceae_Coprobacter',
       'Clostridiales_Family_XIII_AD3011_group',
       'Lachnospiraceae_Shuttleworthia', 'Ruminococcaceae_Oscillospira',
       'Ruminococcaceae_Pseudoflavonifractor', 'Ruminococcaceae_UCG_005',
       'Ruminococcaceae_Ruminococcus_1', 'Eggerthellaceae_Eggerthella',
       'Muribaculaceae_NA',
       'Clostridiaceae_1_Clostridium_sensu_stricto_1',
       'Lachnospiraceae_Coprococcus_3', 'Lachnospiraceae_Hungatella',
       'Lachnospiraceae_ND3007_group', 'Peptococcaceae_Peptococcus',
       'Ruminococcaceae_Ruminiclostridium_6',
       'Ruminococcaceae_NK4A214_group', 'Ruminococcaceae_UCG_003',
       'Clostridiales_NA_NA', 'Burkholderiaceae_Parasutterella',
       'Peptococcaceae_NA', 'Eggerthellaceae_DNF00809',
       'Enterobacteriaceae_Escherichia_Shigella'], dtype=object)

In [22]:
to_plot_activity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24 entries, 771 to 5127
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   dependent_feature  24 non-null     object 
 1   main1              24 non-null     object 
 2   main2              24 non-null     object 
 3   interaction        24 non-null     object 
 4   beta_main1         24 non-null     float64
 5   p_main1            24 non-null     float64
 6   beta_main2         24 non-null     float64
 7   p_main2            24 non-null     float64
 8   beta_interaction   24 non-null     float64
 9   p_interaction      24 non-null     float64
 10  r2_train           24 non-null     float64
 11  r2_test            0 non-null      float64
 12  n_train            24 non-null     float64
 13  n_test             0 non-null      float64
 14  formula            24 non-null     object 
 15  q_interaction      24 non-null     float64
 16  class_tax          24 non-nul

In [23]:
to_plot_activity['beta_main1_sign'] = np.sign(to_plot_activity['beta_main1'])

In [24]:
to_plot_activity.groupby(['dependent_feature']).beta_main1_sign.value_counts()

dependent_feature                    beta_main1_sign
bedtime_int_std_log1p_resid           1.0                5
sleep_awakeDuration_log1p_resid      -1.0                2
sleep_awakeDuration_std_log1p_resid  -1.0               12
                                      1.0                2
sleep_efficiency_std_log1p_resid      1.0                1
sleep_minutesAsleep_std_log1p_resid  -1.0                2
Name: count, dtype: int64

In [25]:
to_plot_activity.groupby(['dependent_feature', 'beta_main1_sign']).main2.unique()

dependent_feature                    beta_main1_sign
bedtime_int_std_log1p_resid           1.0               [Barnesiellaceae_Coprobacter, Clostridiales_Fa...
sleep_awakeDuration_log1p_resid      -1.0               [Ruminococcaceae_UCG_005, Ruminococcaceae_Rumi...
sleep_awakeDuration_std_log1p_resid  -1.0               [Muribaculaceae_NA, Clostridiaceae_1_Clostridi...
                                      1.0               [Eggerthellaceae_Eggerthella, Lachnospiraceae_...
sleep_efficiency_std_log1p_resid      1.0                                             [Peptococcaceae_NA]
sleep_minutesAsleep_std_log1p_resid  -1.0               [Eggerthellaceae_DNF00809, Enterobacteriaceae_...
Name: main2, dtype: object

In [26]:
sorted(to_plot_activity.groupby(['dependent_feature', 'beta_main1_sign']).main2.unique().tolist()[3])

['Eggerthellaceae_Eggerthella', 'Lachnospiraceae_Hungatella']

In [27]:
to_plot_activity.groupby(['dependent_feature', 'beta_main1_sign']).main2.unique().tolist()

[array(['Barnesiellaceae_Coprobacter',
        'Clostridiales_Family_XIII_AD3011_group',
        'Lachnospiraceae_Shuttleworthia', 'Ruminococcaceae_Oscillospira',
        'Ruminococcaceae_Pseudoflavonifractor'], dtype=object),
 array(['Ruminococcaceae_UCG_005', 'Ruminococcaceae_Ruminococcus_1'],
       dtype=object),
 array(['Muribaculaceae_NA',
        'Clostridiaceae_1_Clostridium_sensu_stricto_1',
        'Lachnospiraceae_Coprococcus_3', 'Lachnospiraceae_ND3007_group',
        'Peptococcaceae_Peptococcus',
        'Ruminococcaceae_Ruminiclostridium_6',
        'Ruminococcaceae_NK4A214_group', 'Ruminococcaceae_UCG_003',
        'Ruminococcaceae_UCG_005', 'Ruminococcaceae_Ruminococcus_1',
        'Clostridiales_NA_NA', 'Burkholderiaceae_Parasutterella'],
       dtype=object),
 array(['Eggerthellaceae_Eggerthella', 'Lachnospiraceae_Hungatella'],
       dtype=object),
 array(['Peptococcaceae_NA'], dtype=object),
 array(['Eggerthellaceae_DNF00809',
        'Enterobacteriaceae_Escherichia

In [28]:
to_plot_activity.main2.unique()

array(['Barnesiellaceae_Coprobacter',
       'Clostridiales_Family_XIII_AD3011_group',
       'Lachnospiraceae_Shuttleworthia', 'Ruminococcaceae_Oscillospira',
       'Ruminococcaceae_Pseudoflavonifractor', 'Ruminococcaceae_UCG_005',
       'Ruminococcaceae_Ruminococcus_1', 'Eggerthellaceae_Eggerthella',
       'Muribaculaceae_NA',
       'Clostridiaceae_1_Clostridium_sensu_stricto_1',
       'Lachnospiraceae_Coprococcus_3', 'Lachnospiraceae_Hungatella',
       'Lachnospiraceae_ND3007_group', 'Peptococcaceae_Peptococcus',
       'Ruminococcaceae_Ruminiclostridium_6',
       'Ruminococcaceae_NK4A214_group', 'Ruminococcaceae_UCG_003',
       'Clostridiales_NA_NA', 'Burkholderiaceae_Parasutterella',
       'Peptococcaceae_NA', 'Eggerthellaceae_DNF00809',
       'Enterobacteriaceae_Escherichia_Shigella'], dtype=object)

In [29]:
to_plot_activity[['main2', 'class_tax']]

,main2,class_tax
771,Barnesiellaceae_Coprobacter,Bacteroidia
797,Clostridiales_Family_XIII_AD3011_group,Clostridia
820,Lachnospiraceae_Shuttleworthia,Clostridia
844,Ruminococcaceae_Oscillospira,Clostridia
847,Ruminococcaceae_Pseudoflavonifractor,Clostridia
1311,Ruminococcaceae_UCG_005,Clostridia
1318,Ruminococcaceae_Ruminococcus_1,Clostridia
1669,Eggerthellaceae_Eggerthella,Coriobacteriia
1681,Muribaculaceae_NA,Bacteroidia
1698,Clostridiaceae_1_Clostridium_sensu_stricto_1,Clostridia


In [30]:
to_plot_activity["dependent_feature"] = to_plot_activity["dependent_feature"].replace({
    "sleep_awakeDuration_log1p_resid": "log1p(awake_duration)",
    "sleep_awakeDuration_std_log1p_resid": "log1p(awake_duration_SD)",
    "bedtime_int_std_log1p_resid": "log1p(bedtime_SD)",
    "sleep_efficiency_std_log1p_resid": "log1p(sleep_efficiency_SD)",
    "sleep_minutesAsleep_std_log1p_resid": "log1p(minutes_asleep_SD)",
})

### Open up the sleep happiness scores defined using the OHQ ~ Fitbit regressions to mark whether a microbe present vs absent improves or degrades activity's effect on sleep

In [2]:
sleep_happiness_scores = pd.read_csv('../activity_sleep_happiness_questionnaire_alignment_with_fitbit/fitbit_happiness_sleep_quality_scores_02-23-2026.csv')

In [3]:
sleep_happiness_scores.rename(columns={'beta_sign_sum': 'sleep_quality_sign'}, inplace=True)
sleep_happiness_scores['sleep_quality_sign'] = np.sign(sleep_happiness_scores['sleep_quality_sign'])

In [4]:
sleep_happiness_scores

,dependent_feature,sleep_quality_sign
0,bedtime_int_log1p_resid,-1.0
1,bedtime_int_std_log1p_resid,-1.0
2,sleep_awakeDuration_log1p_resid,-1.0
3,sleep_awakeDuration_std_log1p_resid,-1.0
4,sleep_awakeningsCount_log1p_resid,0.0
5,sleep_awakeningsCount_std_log1p_resid,-1.0
6,sleep_efficiency_log1p_resid,1.0
7,sleep_efficiency_std_log1p_resid,-1.0
8,sleep_minutesAfterWakeup_log1p_resid,-1.0
9,sleep_minutesAfterWakeup_std_log1p_resid,-1.0


### Keep this code to plot the results

In [38]:
import numpy as np
import pandas as pd
import altair as alt

# ------------------------------------------------------------
# 1) Compute microbe-specific sleep delta (binary -1/0/+1)
#    (presence improves vs worsens activity on sleep quality score)
# ------------------------------------------------------------
def compute_sleep_delta(df: pd.DataFrame, sleep_quality_sign_df: pd.DataFrame):
    """
    Returns:
      sleep_delta_df: columns ["main2", "sleep_delta"] where sleep_delta takes on values -1, 0, or 1
      work: row-level table with sleep_sign and scores for debugging
    """
    needed_df = {"dependent_feature", "main2", "beta_main1", "beta_interaction"}
    missing = needed_df - set(df.columns)
    if missing:
        raise ValueError(f"df missing required columns: {sorted(missing)}")

    needed_sign = {"dependent_feature", "sleep_quality_sign"}
    missing = needed_sign - set(sleep_quality_sign_df.columns)
    if missing:
        raise ValueError(f"sleep_quality_sign_df missing required columns: {sorted(missing)}")

    sleep_sign_lookup = dict(
        zip(
            sleep_quality_sign_df["dependent_feature"].astype(str),
            sleep_quality_sign_df["sleep_quality_sign"],
        )
    )

    # mapping used earlier in pipeline (cleaned name -> raw variable name in sleep_merged)
    reverse_map = {
        "log1p(awake_duration)": "sleep_awakeDuration_log1p_resid",
        "log1p(awake_duration_SD)": "sleep_awakeDuration_std_log1p_resid",
        "log1p(bedtime_SD)": "bedtime_int_std_log1p_resid",
        "log1p(sleep_efficiency_SD)": "sleep_efficiency_std_log1p_resid",
        "log1p(minutes_asleep_SD)": "sleep_minutesAsleep_std_log1p_resid",
    }

    work = df.copy()
    work["dependent_feature"] = work["dependent_feature"].astype(str)
    work["main2"] = work["main2"].astype(str)

    # reverse cleaned names ONLY for lookup
    work["sleep_lookup_feature"] = work["dependent_feature"].replace(reverse_map)

    # sign lookup (-1/0/+1)
    work["sleep_sign"] = work["sleep_lookup_feature"].map(sleep_sign_lookup).fillna(0).astype(float)
    work["sleep_sign"] = work["sleep_sign"].apply(lambda v: 1 if v > 0 else (-1 if v < 0 else 0)).astype(int)

    # activity slope absent/present
    work["activity_absent"] = pd.to_numeric(work["beta_main1"], errors="coerce")
    work["activity_present"] = pd.to_numeric(work["beta_main1"], errors="coerce") + pd.to_numeric(
        work["beta_interaction"], errors="coerce"
    )
    work = work.dropna(subset=["activity_absent", "activity_present"]).copy()

    # "sleep score" contributions
    # Use the OHQ defined scores and multiply the activity coefficients when microbe present/vs absent to determine
    # the direction that the activity impacts sleep
    work["score_absent"] = work["activity_absent"] * work["sleep_sign"]
    work["score_present"] = work["activity_present"] * work["sleep_sign"]

    summary = (
        work.groupby("main2", as_index=False)
        .agg(
            absent_score=("score_absent", "sum"),
            present_score=("score_present", "sum"),
        )
    )

    # Sum the absent from present activity sleep scored coefficients to determine what direction
    # microbe present vs absent impacts sleep quality
    summary["sleep_delta"] = np.sign(summary["present_score"] - summary["absent_score"]).astype(int)
    sleep_delta_df = summary[["main2", "sleep_delta"]].copy()

    return sleep_delta_df, work


# ------------------------------------------------------------
# 2) Build long_df (Absent vs Present betas) from interaction output
# ------------------------------------------------------------
def make_steps_presence_long_from_interaction_df(
    df: pd.DataFrame,
    dep_col: str = "dependent_feature",
    main1_col: str = "main1",
    microbe_col: str = "main2",
    beta_steps_col: str = "beta_main1",
    beta_int_col: str = "beta_interaction",
):
    needed = [dep_col, main1_col, microbe_col, beta_steps_col, beta_int_col]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    work = df[needed].dropna().copy()
    for c in [dep_col, main1_col, microbe_col]:
        work[c] = work[c].astype(str)

    work["beta_absent"] = pd.to_numeric(work[beta_steps_col], errors="coerce")
    work["beta_present"] = pd.to_numeric(work[beta_steps_col], errors="coerce") + pd.to_numeric(
        work[beta_int_col], errors="coerce"
    )
    work = work.dropna(subset=["beta_absent", "beta_present"]).copy()

    long_df = pd.concat(
        [
            work[[dep_col, main1_col, microbe_col, "beta_absent"]]
            .rename(columns={"beta_absent": "beta"})
            .assign(presence="Absent"),
            work[[dep_col, main1_col, microbe_col, "beta_present"]]
            .rename(columns={"beta_present": "beta"})
            .assign(presence="Present"),
        ],
        ignore_index=True,
    )

    piv = (
        long_df.pivot_table(
            index=[dep_col, microbe_col],
            columns="presence",
            values="beta",
            aggfunc="mean",
        )
        .reset_index()
    )
    piv["delta_beta"] = piv["Present"] - piv["Absent"]
    piv["sign_flip"] = (np.sign(piv["Present"]) != np.sign(piv["Absent"]))

    long_df = long_df.merge(
        piv[[dep_col, microbe_col, "delta_beta", "sign_flip"]],
        on=[dep_col, microbe_col],
        how="left",
    )

    return long_df


# ------------------------------------------------------------
# 3) Plot: per dependent_feature panel, Absent vs Present dots + right sleep quality strip
# ------------------------------------------------------------
def plot_steps_effect_absent_vs_present_vconcat(
    long_df: pd.DataFrame,
    # IMPORTANT: pass the original interaction df used to make long_df
    interaction_df_for_sleep_delta: pd.DataFrame,
    sleep_quality_sign_df: pd.DataFrame,
    dep_col: str = "dependent_feature",
    microbe_col: str = "main2",
    main1_col: str = "main1",
    width: int = 520,
    strip_width: int = 8,
    height_step: int = 14,
    top_n_microbes_per_sleep: int = 25,
    title: str = "Effect on sleep when microbe is absent vs present (scaled β)",
    show_flip_outline: bool = True,
    beta_scale: float = 1e6,
    section_spacing: int = 10,
    show_x_axis_on_all: bool = False,
    x_pad_frac: float = 0.05,
):
    needed = [dep_col, microbe_col, main1_col, "presence", "beta", "sign_flip", "delta_beta"]
    missing = [c for c in needed if c not in long_df.columns]
    if missing:
        raise ValueError(f"long_df missing columns: {missing}")

    # --- compute microbe sleep_delta and merge into long_df ---
    sleep_delta_df, _debug_work = compute_sleep_delta(interaction_df_for_sleep_delta, sleep_quality_sign_df)
    sleep_delta_df["main2"] = sleep_delta_df["main2"].astype(str)

    dfp = long_df.copy()
    dfp[dep_col] = dfp[dep_col].astype(str)
    dfp[microbe_col] = dfp[microbe_col].astype(str)
    dfp[main1_col] = dfp[main1_col].astype(str)

    dfp = dfp.merge(sleep_delta_df, left_on=microbe_col, right_on="main2", how="left")
    dfp["sleep_delta"] = dfp["sleep_delta"].fillna(0).astype(int)

    # numeric scaling
    dfp["beta_plot"] = pd.to_numeric(dfp["beta"], errors="coerce") * beta_scale
    dfp["delta_beta_plot"] = pd.to_numeric(dfp["delta_beta"], errors="coerce") * beta_scale
    dfp = dfp.dropna(subset=["beta_plot", "delta_beta_plot"]).copy()

    # within each panel order the interaction dot/line plots by greatest magnitude
    # of interaction effect in descending order. Also added a function
    # to just plot the top N microbes if desired
    ranker = dfp.groupby([dep_col, microbe_col], as_index=False)["delta_beta_plot"].first()
    ranker["abs_delta"] = ranker["delta_beta_plot"].abs()

    keep = (
        ranker.sort_values("abs_delta", ascending=False)
        .groupby(dep_col)
        .head(top_n_microbes_per_sleep)[[dep_col, microbe_col]]
    )

    dfp = dfp.merge(keep, on=[dep_col, microbe_col], how="inner")
    if dfp.empty:
        raise ValueError("No rows remain after filtering to top microbes per sleep feature.")

    order_df = (
        ranker.merge(keep, on=[dep_col, microbe_col], how="inner")
        .sort_values([dep_col, "abs_delta"], ascending=[True, False])
        .copy()
    )
    order_df["order_rank"] = order_df.groupby(dep_col).cumcount()
    dfp = dfp.merge(order_df[[dep_col, microbe_col, "order_rank"]], on=[dep_col, microbe_col], how="left")

    sleep_order = list(dfp[dep_col].dropna().unique())

    zero = alt.Chart(pd.DataFrame({"x": [0.0]})).mark_rule(strokeDash=[4, 4]).encode(x="x:Q")

    # Make the right hand sleep quality score present vs absent color strip
    strip_fill_scale = alt.Scale(
        domain=[-1, 0, 1],
        range=["#D75F4C", "transparent", "#3A93C3"],
    )
    
    # Plot the panels
    charts = []
    for i, dep in enumerate(sleep_order):
        dsub = dfp[dfp[dep_col] == dep].copy()
        if dsub.empty:
            continue

        # x-axis formatting
        x_axis = alt.Axis(format=".3f")
        if (not show_x_axis_on_all) and (i < len(sleep_order) - 1):
            x_axis = alt.Axis(format=".3f", labels=False, ticks=False, title=None)

        # dynamic x label
        main1_vals = dsub[main1_col].dropna().unique()
        if len(main1_vals) > 0:
            tokens = str(main1_vals[0]).split("_")
            mid_token = tokens[1] if len(tokens) >= 3 else str(main1_vals[0])
        else:
            mid_token = "feature"
        x_title = f"{mid_token} slope (scaled β × {beta_scale:g})"

        # symmetric x-domain
        m = float(np.nanmax(np.abs(dsub["beta_plot"].values))) if len(dsub) else 1.0
        if (not np.isfinite(m)) or (m <= 0):
            m = 1.0
        m *= (1 + x_pad_frac)

        base = alt.Chart(dsub).encode(
            y=alt.Y(
                f"{microbe_col}:N",
                sort=alt.SortField(field="order_rank", order="ascending"),
                title=None,
                axis=alt.Axis(labelLimit=700),
            ),
            x=alt.X(
                "beta_plot:Q",
                title=x_title if (show_x_axis_on_all or i == len(sleep_order) - 1) else None,
                axis=x_axis,
                scale=alt.Scale(domain=[-m, m], zero=False),
            ),
            color=alt.Color(
                "presence:N",
                title="Microbe status" if i == 0 else None,
                scale=alt.Scale(domain=["Present", "Absent"], range=["#1B7939", "#742881"]),
            ),
            tooltip=[
                alt.Tooltip(dep_col + ":N", title="Sleep feature"),
                alt.Tooltip(main1_col + ":N", title="Main1"),
                alt.Tooltip(microbe_col + ":N", title="Microbe"),
                alt.Tooltip("presence:N", title="Microbe status"),
                alt.Tooltip("beta:Q", title="Main1 β (raw)", format=".6g"),
                alt.Tooltip("beta_plot:Q", title=f"Main1 β × {beta_scale:g}", format=".3f"),
                alt.Tooltip("delta_beta:Q", title="Δβ raw", format=".6g"),
                alt.Tooltip("delta_beta_plot:Q", title=f"Δβ × {beta_scale:g}", format=".3f"),
                alt.Tooltip("sleep_delta:O", title="Δ sleep score sign (-1/0/+1)"),
            ],
        )

        lines = alt.Chart(dsub).mark_line(opacity=0.30, color="gray").encode(
            y=alt.Y(f"{microbe_col}:N", sort=alt.SortField(field="order_rank", order="ascending")),
            x=alt.X("beta_plot:Q"),
            detail=alt.Detail(f"{microbe_col}:N"),
        )

        points = base.mark_point(filled=True, size=70)

        layers = [lines, points, zero]

        if show_flip_outline:
            flip_outline = base.transform_filter(alt.datum.sign_flip == True).mark_point(
                filled=False, size=140, strokeWidth=2, color="black"
            )
            layers.insert(2, flip_outline)

        main_panel = alt.layer(*layers).properties(
            width=width,
            height={"step": height_step},
            title=alt.TitleParams(text=dep, anchor="middle", fontWeight="bold", fontSize=13, offset=2),
        )

        # IMPORTANT: dummy column ensures a constant 1-col strip
        strip_df = dsub[[microbe_col, "order_rank", "sleep_delta"]].copy()
        strip_df["dummy"] = "strip"

        sleep_delta_strip = (
            alt.Chart(strip_df)
            .mark_rect(height=height_step - 2)
            .encode(
                x=alt.X("dummy:N", axis=None),
                y=alt.Y(
                    f"{microbe_col}:N",
                    sort=alt.SortField(field="order_rank", order="ascending"),
                    axis=alt.Axis(labels=False, ticks=False, domain=False, title=None, grid=False),
                ),
                fill=alt.Fill(
                    "sleep_delta:Q",
                    scale=strip_fill_scale,
                    legend=alt.Legend(
                        title="Perceived Sleep Quality",
                        orient="right",
                        direction="vertical",
                        symbolType="square",
                        symbolSize=80,
                        symbolStrokeWidth=0.5,
                        padding=4,
                        labelFontSize=10,
                        titleFontSize=11,
                    ),
                ),
                tooltip=[
                    alt.Tooltip(f"{microbe_col}:N", title="Microbe"),
                    alt.Tooltip("sleep_delta:O", title="Δ sleep score sign (-1/0/+1)"),
                ],
            )
            .properties(width=strip_width, height={"step": height_step})
        )

        charts.append(alt.hconcat(main_panel, sleep_delta_strip, spacing=2).resolve_scale(y="shared"))

    final = (
        alt.vconcat(*charts, spacing=section_spacing)
        .resolve_scale(x="independent", y="independent")
        .properties(title=alt.TitleParams(text=title, anchor="start", fontSize=16))
        .configure_view(stroke=None)
        .configure_axis(grid=True, titleFontWeight="normal")
    )

    return final

### Keep this code to make the taxonomic class color bar that we can overlay onto the main figure in Biorender

In [40]:
import numpy as np
import pandas as pd
import altair as alt

# ------------------------------------------------------------
# 3) Plot: per dependent_feature panel
#    y labels kept intact
#    thin taxonomic class strip on the left next to microbe labels
#    right strip for sleep_delta
# ------------------------------------------------------------
def plot_steps_effect_absent_vs_present_vconcat(
    long_df: pd.DataFrame,
    interaction_df_for_sleep_delta: pd.DataFrame,
    sleep_quality_sign_df: pd.DataFrame,
    dep_col: str = "dependent_feature",
    microbe_col: str = "main2",
    main1_col: str = "main1",
    width: int = 520,
    strip_width: int = 8,
    class_strip_width: int = 5,
    height_step: int = 14,
    top_n_microbes_per_sleep: int = 25,
    title: str = "Effect on sleep when microbe is absent vs present (scaled β)",
    show_flip_outline: bool = True,
    beta_scale: float = 1e6,
    section_spacing: int = 10,
    show_x_axis_on_all: bool = False,
    x_pad_frac: float = 0.05,
):
    needed = [dep_col, microbe_col, main1_col, "presence", "beta", "sign_flip", "delta_beta"]
    missing = [c for c in needed if c not in long_df.columns]
    if missing:
        raise ValueError(f"long_df missing columns: {missing}")

    # --- compute microbe sleep_delta and merge into long_df ---
    sleep_delta_df, _debug_work = compute_sleep_delta(
        interaction_df_for_sleep_delta, sleep_quality_sign_df
    )
    sleep_delta_df["main2"] = sleep_delta_df["main2"].astype(str)

    dfp = long_df.copy()
    dfp[dep_col] = dfp[dep_col].astype(str)
    dfp[microbe_col] = dfp[microbe_col].astype(str)
    dfp[main1_col] = dfp[main1_col].astype(str)

    # bring in class_tax from long_df if present; otherwise from interaction df
    if "class_tax" not in dfp.columns:
        if "class_tax" not in interaction_df_for_sleep_delta.columns:
            raise ValueError("Need a 'class_tax' column in long_df or interaction_df_for_sleep_delta.")
        class_map = (
            interaction_df_for_sleep_delta[[microbe_col, "class_tax"]]
            .dropna(subset=[microbe_col])
            .copy()
        )
        class_map[microbe_col] = class_map[microbe_col].astype(str)
        class_map = class_map.drop_duplicates(subset=[microbe_col])
        dfp = dfp.merge(class_map, on=microbe_col, how="left")

    dfp["class_tax"] = dfp["class_tax"].fillna("Unknown").astype(str)

    dfp = dfp.merge(sleep_delta_df, left_on=microbe_col, right_on="main2", how="left")
    dfp["sleep_delta"] = dfp["sleep_delta"].fillna(0).astype(int)

    # numeric scaling
    dfp["beta_plot"] = pd.to_numeric(dfp["beta"], errors="coerce") * beta_scale
    dfp["delta_beta_plot"] = pd.to_numeric(dfp["delta_beta"], errors="coerce") * beta_scale
    dfp = dfp.dropna(subset=["beta_plot", "delta_beta_plot"]).copy()

    # within each panel order the interaction dot/line plots by greatest magnitude
    # of interaction effect in descending order. Also added a function
    # to just plot the top N microbes if desired
    ranker = dfp.groupby([dep_col, microbe_col], as_index=False)["delta_beta_plot"].first()
    ranker["abs_delta"] = ranker["delta_beta_plot"].abs()

    keep = (
        ranker.sort_values("abs_delta", ascending=False)
        .groupby(dep_col)
        .head(top_n_microbes_per_sleep)[[dep_col, microbe_col]]
    )

    dfp = dfp.merge(keep, on=[dep_col, microbe_col], how="inner")
    if dfp.empty:
        raise ValueError("No rows remain after filtering to top microbes per sleep feature.")

    order_df = (
        ranker.merge(keep, on=[dep_col, microbe_col], how="inner")
        .sort_values([dep_col, "abs_delta"], ascending=[True, False])
        .copy()
    )
    order_df["order_rank"] = order_df.groupby(dep_col).cumcount()
    dfp = dfp.merge(order_df[[dep_col, microbe_col, "order_rank"]], on=[dep_col, microbe_col], how="left")

    sleep_order = list(dfp[dep_col].dropna().unique())

    zero = alt.Chart(pd.DataFrame({"x": [0.0]})).mark_rule(strokeDash=[4, 4]).encode(x="x:Q")

    # right-hand side perceived sleep quality strip color scale
    strip_fill_scale = alt.Scale(
        domain=[-1, 0, 1],
        range=["#D75F4C", "transparent", "#3A93C3"],
    )

    # taxonomic class color pallette
    all_classes = sorted(dfp["class_tax"].dropna().astype(str).unique().tolist())
    class_scale = alt.Scale(
            domain=all_classes,
            range=[
                        "#66C2A5",
                        "#FC8D62",
                        "#8DA0CB",
                        "#6A3D9A",
                        "#A6D854",
                    ]
    )
    
    # Plot panels
    charts = []
    for i, dep in enumerate(sleep_order):
        dsub = dfp[dfp[dep_col] == dep].copy()
        if dsub.empty:
            continue

        # x-axis formatting
        x_axis = alt.Axis(format=".3f")
        if (not show_x_axis_on_all) and (i < len(sleep_order) - 1):
            x_axis = alt.Axis(format=".3f", labels=False, ticks=False, title=None)

        # dynamic x label
        main1_vals = dsub[main1_col].dropna().unique()
        if len(main1_vals) > 0:
            tokens = str(main1_vals[0]).split("_")
            mid_token = tokens[1] if len(tokens) >= 3 else str(main1_vals[0])
        else:
            mid_token = "feature"
        x_title = f"{mid_token} slope (scaled β × {beta_scale:g})"

        # symmetric x-domain
        m = float(np.nanmax(np.abs(dsub["beta_plot"].values))) if len(dsub) else 1.0
        if (not np.isfinite(m)) or (m <= 0):
            m = 1.0
        m *= (1 + x_pad_frac)

        # main panel with visible y labels
        base = alt.Chart(dsub).encode(
            y=alt.Y(
                f"{microbe_col}:N",
                sort=alt.SortField(field="order_rank", order="ascending"),
                title=None,
                axis=alt.Axis(labelLimit=700),
            ),
            x=alt.X(
                "beta_plot:Q",
                title=x_title if (show_x_axis_on_all or i == len(sleep_order) - 1) else None,
                axis=x_axis,
                scale=alt.Scale(domain=[-m, m], zero=False),
            ),
            color=alt.Color(
                "presence:N",
                title="Microbe status" if i == 0 else None,
                scale=alt.Scale(domain=["Present", "Absent"], range=["#1B7939", "#742881"]),
            ),
            tooltip=[
                alt.Tooltip(dep_col + ":N", title="Sleep feature"),
                alt.Tooltip(main1_col + ":N", title="Main1"),
                alt.Tooltip(microbe_col + ":N", title="Microbe"),
                alt.Tooltip("class_tax:N", title="Class"),
                alt.Tooltip("presence:N", title="Microbe status"),
                alt.Tooltip("beta:Q", title="Main1 β (raw)", format=".6g"),
                alt.Tooltip("beta_plot:Q", title=f"Main1 β × {beta_scale:g}", format=".3f"),
                alt.Tooltip("delta_beta:Q", title="Δβ raw", format=".6g"),
                alt.Tooltip("delta_beta_plot:Q", title=f"Δβ × {beta_scale:g}", format=".3f"),
                alt.Tooltip("sleep_delta:O", title="Δ sleep score sign (-1/0/+1)"),
            ],
        )

        lines = alt.Chart(dsub).mark_line(opacity=0.30, color="gray").encode(
            y=alt.Y(f"{microbe_col}:N", sort=alt.SortField(field="order_rank", order="ascending")),
            x=alt.X("beta_plot:Q"),
            detail=alt.Detail(f"{microbe_col}:N"),
        )

        points = base.mark_point(filled=True, size=70)

        layers = [lines, points, zero]

        if show_flip_outline:
            flip_outline = base.transform_filter(alt.datum.sign_flip == True).mark_point(
                filled=False, size=140, strokeWidth=2, color="black"
            )
            layers.insert(2, flip_outline)

        main_panel = alt.layer(*layers).properties(
            width=width,
            height={"step": height_step},
            title=alt.TitleParams(text=dep, anchor="middle", fontWeight="bold", fontSize=13, offset=2),
        )

        # thin class strip on left hand side: separate chart
        # keeps y labels intact on main panel
        class_df = dsub[[microbe_col, "order_rank", "class_tax"]].drop_duplicates().copy()
        class_df["dummy"] = "class_strip"

        class_strip = (
            alt.Chart(class_df)
            .mark_rect(height=height_step - 2)
            .encode(
                x=alt.X("dummy:N", axis=None),
                y=alt.Y(
                    f"{microbe_col}:N",
                    sort=alt.SortField(field="order_rank", order="ascending"),
                    axis=alt.Axis(labels=False, ticks=False, domain=False, title=None, grid=False),
                ),
                fill=alt.Fill(
                    "class_tax:N",
                    scale=class_scale,
                    legend=alt.Legend(
                        title="Microbe Class",
                        orient="right",
                        direction="vertical",
                        symbolType="square",
                        symbolSize=80,
                        symbolStrokeWidth=0.5,
                        padding=4,
                        labelFontSize=10,
                        titleFontSize=11,
                    ) if i == 0 else None,
                ),
                tooltip=[
                    alt.Tooltip(f"{microbe_col}:N", title="Microbe"),
                    alt.Tooltip("class_tax:N", title="Class"),
                ],
            )
            .properties(width=class_strip_width, height={"step": height_step})
        )

        # right hand side color strip: perceived sleep quality change from absent to present (sleep_delta)

        strip_df = dsub[[microbe_col, "order_rank", "sleep_delta"]].drop_duplicates().copy()
        strip_df["dummy"] = "strip"

        sleep_delta_strip = (
            alt.Chart(strip_df)
            .mark_rect(height=height_step - 2)
            .encode(
                x=alt.X("dummy:N", axis=None),
                y=alt.Y(
                    f"{microbe_col}:N",
                    sort=alt.SortField(field="order_rank", order="ascending"),
                    axis=alt.Axis(labels=False, ticks=False, domain=False, title=None, grid=False),
                ),
                fill=alt.Fill(
                    "sleep_delta:Q",
                    scale=strip_fill_scale,
                    legend=None
                ),
                tooltip=[
                    alt.Tooltip(f"{microbe_col}:N", title="Microbe"),
                    alt.Tooltip("sleep_delta:O", title="Δ sleep score sign (-1/0/+1)"),
                ],
            )
            .properties(width=strip_width, height={"step": height_step})
        )

        # append all the charts together: labels | thin class strip | main plot | right strip
        charts.append(
            alt.hconcat(
                class_strip,
                main_panel,
                sleep_delta_strip,
                spacing=2
            ).resolve_scale(y="shared")
        )

    final = (
        alt.vconcat(*charts, spacing=section_spacing)
        .resolve_scale(x="independent", y="independent")
        .properties(title=alt.TitleParams(text=title, anchor="start", fontSize=16))
        .configure_view(stroke=None)
        .configure_axis(grid=True, titleFontWeight="normal")
    )

    return final

In [41]:
long_df = make_steps_presence_long_from_interaction_df(
    to_plot_activity,
    dep_col="dependent_feature",
    main1_col="main1",
    microbe_col="main2",
    beta_steps_col="beta_main1",
    beta_int_col="beta_interaction",
)

chart = plot_steps_effect_absent_vs_present_vconcat(
    long_df=long_df,
    interaction_df_for_sleep_delta=to_plot_activity,
    sleep_quality_sign_df=sleep_happiness_scores,
    width=520,
    strip_width=8,
    height_step=14,
    top_n_microbes_per_sleep=1000, # Plot all the hits, added this in case there were a lot of hits that we didn't want to visualize all at once
    beta_scale=1000,
    section_spacing=15,
    show_x_axis_on_all=True,
    title="Effect on sleep when microbe is absent vs present (scaled β)",
    show_flip_outline=False,
)

chart

alt.VConcatChart(...)